1. dividing the data to bins/sliding windows - creating coef_df - run if not using the coef_df created in step9

In [8]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from tqdm import tqdm
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIGURATION ---
min_age = 20
max_age = 100

# Options: 'fixed', 'sliding_n_subjects', or 'sliding_n_years'
STRATEGY = 'sliding_n_years' 

# Parameters for 'sliding_n_subjects'
window_subjects = 100 
step_subjects = 50

# Parameters for 'sliding_n_years'
window_years = 5
step_years = 1

# Parameters for 'fixed'
bin_size = 5

In [9]:



# --- 1. DATA LOADING & PREP ---
combined_df = pd.read_pickle('/home/gaia/Projects/legacy_data/combined_gm_volumes.pkl')
volumes = combined_df[(combined_df['classification_label'] == 1) | (combined_df['source'] == 'snbb')].copy()
volumes['age_in_years'] = pd.to_numeric(volumes['age_in_years'], errors='coerce')
volumes = volumes[(volumes['age_in_years'] >= min_age) & (volumes['age_in_years'] <= max_age)]
volumes = volumes.dropna(subset=['age_in_years', 'birth_year', 'sex', 'tiv'])
# --- 2. WINDOW DEFINITION ---
mappings = []

# Prepare sorted metadata for sliding strategies
meta = (
    volumes[['session_id', 'age_in_years']]
    .drop_duplicates('session_id')
    .sort_values('age_in_years')
    .reset_index(drop=True)
)

if STRATEGY == 'fixed':
    bins = np.arange(min_age, max_age + bin_size, bin_size)
    volumes['bin'] = pd.cut(volumes['age_in_years'], bins=bins)
    volumes = volumes.dropna(subset=['bin'])

elif STRATEGY == 'sliding_n_subjects':
    session_ids = meta['session_id'].values
    for i in range(0, len(session_ids) - window_subjects + 1, step_subjects):
        subset_ids = session_ids[i : i + window_subjects]
        age_min, age_max = meta.iloc[i]['age_in_years'], meta.iloc[i + window_subjects - 1]['age_in_years']
        label = f"W{i//step_subjects:02d}_Age_{age_min:.1f}-{age_max:.1f}"
        for sid in subset_ids:
            mappings.append({'session_id': sid, 'bin': label})

elif STRATEGY == 'sliding_n_years':
    min_age = meta['age_in_years'].min()
    max_age = meta['age_in_years'].max()
    
    # Iterate through age ranges
    current_start = min_age
    window_idx = 0
    while (current_start + window_years) <= max_age:
        current_end = current_start + window_years
        
        # Find session IDs within this specific age range
        mask = (meta['age_in_years'] >= current_start) & (meta['age_in_years'] < current_end)
        subset_ids = meta.loc[mask, 'session_id'].values
        
        if len(subset_ids) > 0:
            label = f"W{window_idx:02d}_Age_{current_start:.1f}-{current_end:.1f}"
            for sid in subset_ids:
                mappings.append({'session_id': sid, 'bin': label})
        
        current_start += step_years
        window_idx += 1

# Merge back for all sliding strategies
if 'sliding' in STRATEGY:
    map_df = pd.DataFrame(mappings)
    volumes = volumes.merge(map_df, on='session_id')

# --- 3. ANALYSIS ---
atlas_csv = pd.read_csv("/home/gaia/Projects/legacy_data/my_master/space-MNI152_atlas-schaefer2018tian2020_res-1mm_den-400_div-7networks_dseg.csv")
results = []

# Grouping by 'bin' and 'region_label' handles all logic in one pass
grouped = volumes.groupby(['bin', 'region_label'], observed=True)

for (bin_label, roi), df_roi in tqdm(grouped, desc="Processing ROI per Window"):
    # Ensure sufficient sample size for the model
    if len(df_roi) < 20: 
        continue
        
    try:
        # GMV_ROI_1 ~ b1*birth_year + covariates
        model = smf.ols(
            'volume_mm3 ~ birth_year + C(sex) + tiv + age_in_years', 
            data=df_roi
        ).fit()
        
        for var in model.params.index:
            results.append({
                'bin': bin_label,
                'region_label': roi,
                'variable': var,
                'coef': model.params[var],
                't': model.tvalues[var],
                'p': model.pvalues[var],
                'n_subjects': df_roi['session_id'].nunique()
            })
    except:
        continue

# --- 4. POST-PROCESSING ---
coef_df = pd.DataFrame(results)

# Add Atlas names
coef_df = coef_df.merge(atlas_csv[['index', 'name']], left_on='region_label', right_on='index', how='left')
coef_df.rename(columns={'name': 'region_name'}, inplace=True)

# Multiple Comparison Correction (FDR) per window for 'birth_year'
final_rows = []
for label, group in coef_df.groupby('bin'):
    mask = group['variable'] == 'birth_year'
    if mask.any():
        _, fdr_p, _, _ = multipletests(group.loc[mask, 'p'], method='fdr_bh')
        group.loc[mask, 'fdr_p'] = fdr_p
    final_rows.append(group)

coef_df = pd.concat(final_rows)

# --- 5. SAVE ---
suffix = f"sliding_ws{window_subjects}_ss{step_subjects}" if STRATEGY == 'sliding_n_subjects' else f"sliding_wy{window_years}_sy{step_years}" if STRATEGY == 'sliding_n_years' else f"fixed_bin{bin_size}"

print("Analysis Complete.")

Processing ROI per Window: 100%|██████████| 30418/30418 [02:07<00:00, 238.93it/s]  


Analysis Complete.


In [ ]:
# coef_df.to_csv(f"/home/gaia/Projects/legacy_data/legacy_pipe/data/interim/coef_df_{suffix}.csv", index=False)


# QA - comparing the result here to step9

In [11]:
# # compare the coef_df that created here to the one created in step9
# step9_coef_df = pd.read_csv("/home/gaia/Projects/legacy_data/legacy_pipe/data/interim/coef_df_age_bins_size_5.csv")

# step12_coef_df = coef_df.copy()  # Use the coef_df created in this step


# # remove the columns from step12_coef_df: n_subjects, index
# step12_coef_df = step12_coef_df.drop(columns=['n_subjects', 'index'], errors='ignore')


# # Define the specific interval to remove
# bin_to_remove = pd.Interval(15, 20, closed='right')

# # Filter it out
# step12_coef_df = step12_coef_df[~step12_coef_df['bin'].isin([bin_to_remove])]


# # order the columns 
# step12_coef_df = step12_coef_df[[ 'bin', 'region_label', 'region_name', 'variable', 'coef', 't', 'p', 'fdr_p']]
# step9_coef_df = step9_coef_df[['age_bin', 'region_label', 'region_name', 'variable', 'coef', 't', 'p', 'fdr_p']]

# # order both by region_label, region_name, variable
# step9_coef_df = step9_coef_df.sort_values(['region_label', 'region_name', 'variable'])
# step12_coef_df = step12_coef_df.sort_values(['region_label', 'region_name', 'variable'])



# print(f"are they the same? {step9_coef_df.equals(step12_coef_df)}")

# print(f"age bins: step9: {step9_coef_df['age_bin'].unique()}, step12: {step12_coef_df['bin'].unique()}")